# Logistics Delay Prediction ? SageMaker Training

**Cloud Computing ? Modular Assignment 6**

## Objective
Train and evaluate a logistics delay prediction model using Amazon SageMaker in the eu-north-1 AWS Region.

## Data Sources
1. Synthetic shipment records (1,000 shipments).
2. Historical weather observations retrieved from Open-Meteo.

## Workflow
Data validation ? Amazon S3 ? SageMaker training ? Model evaluation ? Model deployment ? Real-time inference ? Monitoring.

**Status:** Local preparation completed; AWS training quota approval pending.


## 1. Dataset Validation

Load the preprocessed shipment-weather dataset and verify its structure, completeness, and consistency before cloud training.


In [ ]:
from pathlib import Path
import pandas as pd

# Locate the project dataset from the notebook directory or project root.
candidates = [
    Path("../data/processed/ml_dataset.csv"),
    Path("data/processed/ml_dataset.csv")
]

dataset_path = next((p for p in candidates if p.exists()), None)

if dataset_path is None:
    raise FileNotFoundError("Processed dataset not found.")

df = pd.read_csv(dataset_path)

print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Columns:", df.columns.tolist())

assert len(df) == 1000, "Unexpected shipment count"
assert df.shape[1] == 16, "Unexpected column count"
assert not df.isna().any().any(), "Missing values detected"
assert not df.duplicated().any(), "Duplicate records detected"

print("Dataset validation: PASSED")
display(df.head())


## 2. AWS SageMaker Configuration

Configure the AWS Region, S3 input dataset, and training output location. Training will be launched only after the required SageMaker quota and IAM permissions are verified.


In [ ]:
# SageMaker training configuration
AWS_REGION = "eu-north-1"
S3_BUCKET = "logistics-delay-m6-habib"

S3_TRAINING_DATA = f"s3://{S3_BUCKET}/data/ml_dataset.csv"
S3_MODEL_OUTPUT = f"s3://{S3_BUCKET}/model-artifacts/"

TRAINING_INSTANCE = "ml.m5.large"
TRAINING_INSTANCE_COUNT = 1

print("AWS Region:", AWS_REGION)
print("Training dataset:", S3_TRAINING_DATA)
print("Model output:", S3_MODEL_OUTPUT)
print("Training instance:", TRAINING_INSTANCE)
print("Instance count:", TRAINING_INSTANCE_COUNT)
print("Configuration prepared. No AWS resources launched.")


## 3. SageMaker Estimator Preparation

This cell only configures the estimator. It does not start a training job.


In [ ]:
from pathlib import Path
import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

# Self-contained configuration: this cell does not depend on earlier cells.
AWS_REGION = "eu-north-1"
S3_BUCKET = "logistics-delay-m6-habib"
EXECUTION_ROLE_ARN = "arn:aws:iam::345370629917:role/Logistics-M6-SageMaker-ExecutionRole"
S3_TRAINING_DATA = f"s3://{S3_BUCKET}/data/ml_dataset.csv"
S3_MODEL_OUTPUT = f"s3://{S3_BUCKET}/model-artifacts/"

# Find the training script whether VS Code starts in the project root
# or in the notebooks directory.
source_dir = next(
    (path for path in (Path("deployment/source"), Path("../deployment/source"))
     if (path / "train.py").is_file()),
    None,
)
if source_dir is None:
    raise FileNotFoundError(
        "Cannot find deployment/source/train.py. Open the project folder in VS Code."
    )

session = sagemaker.Session(
    boto_session=boto3.Session(region_name=AWS_REGION)
)

estimator = SKLearn(
    entry_point="train.py",
    source_dir=str(source_dir.resolve()),
    role=EXECUTION_ROLE_ARN,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.4-2",
    output_path=S3_MODEL_OUTPUT,
    base_job_name="logistics-delay-m6-training",
    sagemaker_session=session,
)

print("Estimator configuration: READY")
print("AWS Region:", AWS_REGION)
print("Training script:", source_dir / "train.py")
print("Training dataset:", S3_TRAINING_DATA)
print("Model output:", S3_MODEL_OUTPUT)
print("Training instance: ml.m5.large")
print("Framework version: 1.4-2")
print("Status: NO TRAINING JOB LAUNCHED")
